In [ ]:
import geopandas as gpd
import pandas as pd

from icefabric.cli import get_catalog
from icefabric.helpers import load_creds
from pathlib import Path
import polars as pl
from pyiceberg.expressions import EqualTo, In

import os
from pyiceberg.catalog import load_catalog
from icefabric.builds import load_upstream_json
from icefabric.helpers import load_creds, load_pyiceberg_config
from icefabric.hydrofabric import subset_hydrofabric
from icefabric.schemas import IdType
from icefabric.ui import create_time_series_widget, get_hydrofabric_gages, get_streamflow_data

# Changes the current working dir to be the project root
current_working_dir = Path.cwd()
os.chdir(Path.cwd() / "../")
print(
    f"Changed current working dir from {current_working_dir} to: {Path.cwd()}. This must run at the project root"
)


# dir is where the .env file is located
load_creds(dir=Path.cwd())

# Loading the local pyiceberg config settings
pyiceberg_config = load_pyiceberg_config(Path.cwd())
catalog = load_catalog(
    name="sql",
    type=pyiceberg_config["catalog"]["sql"]["type"],
    uri=pyiceberg_config["catalog"]["sql"]["uri"],
    warehouse=pyiceberg_config["catalog"]["sql"]["warehouse"],
)

Changed current working dir from /home/quercus.hamlin/Documents/code/icefabric-1/examples to: /home/quercus.hamlin/Documents/code/icefabric-1. This must run at the project root


In [23]:
from pyiceberg.expressions import EqualTo, In

In [40]:
gdf_us = gpd.read_file('../data/vpu1.gpkg',layer='network')
gdf_us_nex = gpd.read_file('../data/vpu1.gpkg',layer='nexus')
gdf_m = gpd.read_file('/home/quercus.hamlin/Documents/data/01_nextgen.gpkg', layer='network')
gdf_m_nex = gpd.read_file('/home/quercus.hamlin/Documents/data/01_nextgen.gpkg', layer='nexus')

In [43]:
gdf_m_nex = gdf_m_nex.fillna('na')
gdf_m_nex.loc[gdf_m_nex['id'].str.contains('cnx'), :]

,id,toid,type,vpuid,poi_id,geometry
10981,cnx-276,wb-0,coastal,01,na,POINT (2219022.856 2732684.999)
10982,cnx-275,wb-0,coastal,01,na,POINT (2229518.57 2742509.998)
10983,cnx-274,wb-0,coastal,01,na,POINT (2212231.5 2726640)
10984,cnx-273,wb-0,coastal,01,na,POINT (2244600 2783279.999)
10985,cnx-272,wb-0,coastal,01,na,POINT (2246392.501 2772660.003)
...,...,...,...,...,...,...
11252,cnx-5,wb-0,coastal,01,na,POINT (1946015.154 2272859.997)
11253,cnx-4,wb-0,coastal,01,na,POINT (1865508.752 2233140.002)
11254,cnx-3,wb-0,coastal,01,na,POINT (1874002.497 2238330.003)
11255,cnx-2,wb-0,coastal,01,na,POINT (1851141.345 2220479.997)


In [47]:
gdf_us = gdf_us.fillna('na')
gdf_us.loc[gdf_us['toid'].str.contains('cnx'), :]

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
97030,na,cnx-276,cat-276,na,na,na,NOAA Reference Fabric,na,0.0,27.454477,27.454477,coastal,01,na,na,na,fl-nex,na,na
97031,na,cnx-275,cat-275,na,na,na,NOAA Reference Fabric,na,0.0,25.207642,25.207642,coastal,01,na,na,na,fl-nex,na,na
97032,na,cnx-274,cat-274,na,na,na,NOAA Reference Fabric,na,0.0,16.707601,16.707601,coastal,01,na,na,na,fl-nex,na,na
97033,na,cnx-273,cat-273,na,na,na,NOAA Reference Fabric,na,0.0,32.822992,32.822992,coastal,01,na,na,na,fl-nex,na,na
97034,na,cnx-272,cat-272,na,na,na,NOAA Reference Fabric,na,0.0,78.416994,78.416994,coastal,01,na,na,na,fl-nex,na,na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97301,na,cnx-5,cat-5,na,na,na,NOAA Reference Fabric,na,0.0,0.510302,0.510302,coastal,01,na,na,na,fl-nex,na,na
97302,na,cnx-4,cat-4,na,na,na,NOAA Reference Fabric,na,0.0,4.032448,4.032448,coastal,01,na,na,na,fl-nex,na,na
97303,na,cnx-3,cat-3,na,na,na,NOAA Reference Fabric,na,0.0,0.00225,0.00225,coastal,01,na,na,na,fl-nex,na,na
97304,na,cnx-2,cat-2,na,na,na,NOAA Reference Fabric,na,0.0,1.3815,1.3815,coastal,01,na,na,na,fl-nex,na,na


In [46]:
gdf_m = gdf_m.fillna('na')
gdf_m.loc[gdf_m['toid'].str.contains('cnx'), :]

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
95442,na,cnx-276,cat-276,na,na,na,NOAA Reference Fabric,na,0.0,27.454477,27.454477,coastal,01,na,na,na,fl-nex,na,na
95443,na,cnx-275,cat-275,na,na,na,NOAA Reference Fabric,na,0.0,25.207642,25.207642,coastal,01,na,na,na,fl-nex,na,na
95444,na,cnx-274,cat-274,na,na,na,NOAA Reference Fabric,na,0.0,16.707601,16.707601,coastal,01,na,na,na,fl-nex,na,na
95445,na,cnx-273,cat-273,na,na,na,NOAA Reference Fabric,na,0.0,32.822992,32.822992,coastal,01,na,na,na,fl-nex,na,na
95446,na,cnx-272,cat-272,na,na,na,NOAA Reference Fabric,na,0.0,78.416994,78.416994,coastal,01,na,na,na,fl-nex,na,na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95713,na,cnx-5,cat-5,na,na,na,NOAA Reference Fabric,na,0.0,0.510302,0.510302,coastal,01,na,na,na,fl-nex,na,na
95714,na,cnx-4,cat-4,na,na,na,NOAA Reference Fabric,na,0.0,4.032448,4.032448,coastal,01,na,na,na,fl-nex,na,na
95715,na,cnx-3,cat-3,na,na,na,NOAA Reference Fabric,na,0.0,0.00225,0.00225,coastal,01,na,na,na,fl-nex,na,na
95716,na,cnx-2,cat-2,na,na,na,NOAA Reference Fabric,na,0.0,1.3815,1.3815,coastal,01,na,na,na,fl-nex,na,na


In [29]:
test= gdf_us.loc[gdf_us.type == 'nexus', :]
gdf_us_nex.merge(test, on='id', how='inner').to_file('../data/nex_merge_network.gpkg')

In [6]:
gdf_merge = gdf_us.merge(gdf_m, on=gdf_us.columns.tolist(), how='left')

In [34]:
gdf_merge = gdf_merge.fillna('na')
gdf_merge.loc[gdf_merge['id'].str.contains('inx'), :]

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri


In [12]:
tail = gdf_us.iloc[95715:]

In [19]:
test = gdf_m.iloc[94715:95715]
test

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
94715,nex-9942,wb-9942,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
94716,nex-9943,wb-9943,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
94717,nex-9944,wb-9944,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
94718,nex-9945,wb-9945,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
94719,nex-9946,wb-9946,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95710,None,cnx-8,cat-8,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.567451,0.567451,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
95711,None,cnx-7,cat-7,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.002700,0.002700,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
95712,None,cnx-6,cat-6,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.251100,0.251100,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
95713,None,cnx-5,cat-5,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.510302,0.510302,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None


In [10]:
gdf_m.shape

(95718, 19)

In [22]:
tail

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
95715,nex-8894,wb-8894,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
95716,nex-8899,wb-8899,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
95717,nex-890,wb-890,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
95718,nex-8900,wb-8900,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
95719,nex-8901,wb-8901,None,NaN,NaN,NaN,NOAA Reference Fabric,NaN,NaN,NaN,NaN,nexus,01,NaN,NaN,NaN,fl-nex,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97301,None,cnx-5,cat-5,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.510302,0.510302,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
97302,None,cnx-4,cat-4,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,4.032448,4.032448,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
97303,None,cnx-3,cat-3,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,0.002250,0.002250,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
97304,None,cnx-2,cat-2,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.0,1.381500,1.381500,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None


In [14]:
gdf = gdf_us.merge(gdf_m, on=gdf_us.columns.tolist(), how='inner')
gdf

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
0,wb-20479,tnx-1000000697,cat-20479,NaN,2613603.0,20277.0,NOAA Reference Fabric,4599061.0,1.445138,5.887350,42.698700,terminal,01,2613604.0,0.793282,2613603.0,fl-nex,NaN,None
1,wb-20479,tnx-1000000697,cat-20479,NaN,2613603.0,20277.0,NOAA Reference Fabric,4599715.0,1.445138,5.887350,42.698700,terminal,01,2613605.0,0.205191,2613603.0,fl-nex,NaN,None
2,wb-20479,tnx-1000000697,cat-20479,NaN,2613603.0,20277.0,NOAA Reference Fabric,166196261.0,1.445138,5.887350,42.698700,terminal,01,2613603.0,0.446664,2613603.0,fl-nex,NaN,None
3,wb-18798,tnx-1000000690,cat-18798,NaN,2056798.0,20109.0,NOAA Reference Fabric,7733111.0,0.221200,0.019800,0.019800,terminal,01,2056801.0,0.056196,2056798.0,fl-nex,NaN,None
4,wb-18798,tnx-1000000690,cat-18798,NaN,2056798.0,20109.0,NOAA Reference Fabric,7733105.0,0.221200,0.019800,0.019800,terminal,01,2056800.0,0.123110,2056798.0,fl-nex,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94190,None,cnx-5,cat-5,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,0.510302,0.510302,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
94191,None,cnx-4,cat-4,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,4.032448,4.032448,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
94192,None,cnx-3,cat-3,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,0.002250,0.002250,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None
94193,None,cnx-2,cat-2,NaN,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,1.381500,1.381500,coastal,01,NaN,NaN,NaN,fl-nex,NaN,None


In [15]:
gdf_m.shape

(95718, 19)

In [16]:
gdf_us.shape

(97306, 19)

In [28]:
test = gdf_us_nex.merge(tail, left_on='id', right_on='id', how='inner')
test.to_file('../data/tail.gpkg')

In [27]:
gdf_us_nex.shape

(10980, 6)

In [6]:
nexus_table = catalog.load_table(f"conus_hf.nexus").scan().to_polars()
network_table = catalog.load_table(f"conus_hf.network").scan().to_polars()

In [8]:
nexus_table.filter(pl.col("id") == 'cnx-276')

id,toid,type,vpuid,poi_id,geometry
str,str,str,str,f64,binary
"""cnx-276""","""wb-0""","""coastal""","""01""",null,"b""\x01\x01\x00\x00\x00\xb2\xbf\x87m\x07\xee@A\x89\x07\xdb\x7fF\xd9DA"""


In [20]:
nexus_table.filter(pl.col("id").str.contains('inx'))

id,toid,type,vpuid,poi_id,geometry
str,str,str,str,f64,binary
"""inx-410946""","""wb-0""","""internal""","""03S""",null,"b""\x01\x01\x00\x00\x00b\xcc\x00\xc0\x80e3A\xef\x03\x93\xfeS\xcf(A"""
"""inx-410945""","""wb-0""","""internal""","""03S""",null,"b""\x01\x01\x00\x00\x00g).\x90qE3A\x048\xa8\x01\x98\xc5*A"""
"""inx-410944""","""wb-0""","""internal""","""03S""",null,"b""\x01\x01\x00\x00\x00AU\xd9\x9f\x0cn2AJ\xbd\x7f\xff\xb9\xcf+A"""
"""inx-410943""","""wb-0""","""internal""","""03S""",null,"b""\x01\x01\x00\x00\x00\xdc\xee\xba_d\x184AY\xf0\xfe\xff\xab,(A"""
"""inx-410942""","""wb-0""","""internal""","""03S""",null,"b""\x01\x01\x00\x00\x00\xfc\xed\x82\x7f\xe8\xa04A\x1e\xa9\xcd\x00J\xbf$A"""
…,…,…,…,…,…
"""inx-3299725""","""wb-0""","""internal""","""18""",null,"b""\x01\x01\x00\x00\x00\x84\x88[\x00\x01;>\xc1\x19S\xb4\x80\xfeBAA"""
"""inx-3299724""","""wb-0""","""internal""","""18""",null,"b""\x01\x01\x00\x00\x00\xc0\xfa\xc6\xe0\x88Z>\xc1\xee\xb2\x82\x00'}@A"""
"""inx-3299723""","""wb-0""","""internal""","""18""",null,"b""\x01\x01\x00\x00\x00\x94\x8b\xd2\xffJ\x87>\xc1`tZ\x00""\x0a@A"""


In [ ]:
network_table.filter(pl.col("toid").str.contains('cnx'))

id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
str,str,str,f64,f64,f64,str,f64,f64,f64,f64,str,str,f64,f64,f64,str,f64,str
null,"""cnx-86370""","""cat-86370""",null,null,null,"""NOAA Reference Fabric""",null,0.0,23.554788,23.554788,"""coastal""","""02""",null,null,null,"""fl-nex""",null,null
null,"""cnx-86369""","""cat-86369""",null,null,null,"""NOAA Reference Fabric""",null,0.0,0.435151,0.435151,"""coastal""","""02""",null,null,null,"""fl-nex""",null,null
null,"""cnx-86368""","""cat-86368""",null,null,null,"""NOAA Reference Fabric""",null,0.0,1.700999,1.700999,"""coastal""","""02""",null,null,null,"""fl-nex""",null,null
null,"""cnx-86367""","""cat-86367""",null,null,null,"""NOAA Reference Fabric""",null,0.0,0.451353,0.451353,"""coastal""","""02""",null,null,null,"""fl-nex""",null,null
null,"""cnx-86366""","""cat-86366""",null,null,null,"""NOAA Reference Fabric""",null,0.0,0.103502,0.103502,"""coastal""","""02""",null,null,null,"""fl-nex""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,"""cnx-246664""","""cat-246664""",null,null,null,"""NOAA Reference Fabric""",null,0.0,0.024301,0.024301,"""coastal""","""03N""",null,null,null,"""fl-nex""",null,null
null,"""cnx-246663""","""cat-246663""",null,null,null,"""NOAA Reference Fabric""",null,0.0,0.0873,0.0873,"""coastal""","""03N""",null,null,null,"""fl-nex""",null,null
null,"""cnx-246662""","""cat-246662""",null,null,null,"""NOAA Reference Fabric""",null,0.0,1.592104,1.592104,"""coastal""","""03N""",null,null,null,"""fl-nex""",null,null


In [24]:
network = catalog.load_table(f"conus_hf.network").scan(row_filter=EqualTo('vpuid', '03S')).to_polars()

In [31]:
dropd = network.unique()
dropd.shape
net_us = dropd.to_pandas()

In [29]:
net_m = gpd.read_file('/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_r/vpus/03S_nextgen.gpkg', layer='network')

In [35]:
net_m

,id,toid,divide_id,ds_id,mainstem,hydroseq,hf_source,hf_id,lengthkm,areasqkm,tot_drainage_areasqkm,type,vpuid,hf_hydroseq,hf_lengthkm,hf_mainstem,topo,poi_id,hl_uri
0,wb-423666,tnx-1000002837,cat-423666,NaN,2367036.0,13772.0,NOAA Reference Fabric,16918808.0,4.261507,7.439400,1033.645502,terminal,03S,2367036.0,7.742660,2367036.0,fl-nex,NaN,None
1,wb-423787,tnx-1000002836,cat-423787,NaN,2367331.0,13771.0,NOAA Reference Fabric,16927768.0,0.625780,6.075450,6.075450,terminal,03S,2367331.0,0.625780,2367331.0,fl-nex,NaN,None
2,wb-423786,tnx-1000002835,cat-423786,NaN,2367329.0,13770.0,NOAA Reference Fabric,16924548.0,1.430017,0.626850,0.626850,terminal,03S,2367330.0,1.054026,2367329.0,fl-nex,NaN,None
3,wb-423786,tnx-1000002835,cat-423786,NaN,2367329.0,13770.0,NOAA Reference Fabric,16927766.0,1.430017,0.626850,0.626850,terminal,03S,2367329.0,0.375991,2367329.0,fl-nex,NaN,None
4,wb-420336,tnx-1000002834,cat-420336,NaN,2355291.0,13767.0,NOAA Reference Fabric,16644100.0,2.823140,17.913150,187.740900,terminal,03S,2355291.0,2.823140,2355291.0,fl-nex,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67139,None,inx-410918,cat-410918,412497.0,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,0.613348,0.613348,internal,03S,NaN,NaN,NaN,fl-nex,NaN,None
67140,None,inx-410917,cat-410917,412503.0,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,0.274050,0.274050,internal,03S,NaN,NaN,NaN,fl-nex,NaN,None
67141,None,inx-410916,cat-410916,414196.0,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,0.225451,0.225451,internal,03S,NaN,NaN,NaN,fl-nex,NaN,None
67142,None,inx-410915,cat-410915,413321.0,NaN,NaN,NOAA Reference Fabric,NaN,0.000000,74.188353,74.188353,internal,03S,NaN,NaN,NaN,fl-nex,NaN,None


In [42]:
poi_us = pd.Series(net_us['poi_id'].unique())
poi_m = pd.Series(net_m['poi_id'].unique())
poi_dif = poi_us.loc[~poi_us.isin(poi_m)]

In [44]:
df_us_poi = net_us.loc[net_us['poi_id'].isin(poi_dif)]

In [48]:
df_us_nex = gpd.read_file('./data/vpu3s_new2.gpkg', layer='nexus')

In [50]:
df_us_poi.columns, df_us_nex.columns

(Index(['id', 'toid', 'divide_id', 'ds_id', 'mainstem', 'hydroseq', 'hf_source',
        'hf_id', 'lengthkm', 'areasqkm', 'tot_drainage_areasqkm', 'type',
        'vpuid', 'hf_hydroseq', 'hf_lengthkm', 'hf_mainstem', 'topo', 'poi_id',
        'hl_uri'],
       dtype='object'),
 Index(['id', 'toid', 'type', 'vpuid', 'poi_id', 'geometry'], dtype='object'))

In [52]:
df_us_nex_poi_dif = df_us_nex.merge(df_us_poi, left_on='id', right_on='toid', how='inner')
df_us_nex_poi_dif.to_file('./data/3s_dif.gpkg')

In [54]:
df_us_nex = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_us3/vpu01.gpkg", layer="nexus")
df_m_nex = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_r/vpus/01_nextgen.gpkg", layer="nexus")

In [55]:
df_us_nex.shape, df_m_nex.shape

((11256, 6), (11257, 6))

In [56]:
df_us_nex

,id,toid,type,vpuid,poi_id,geometry
0,nex-1000,wb-1000,nexus,01,None,POINT (2065964.551 2865305.933)
1,nex-10000,wb-10000,nexus,01,23011,POINT (2028886.179 2457811.216)
2,nex-10004,wb-10004,nexus,01,22988,POINT (2031522.75 2455486.067)
3,nex-10008,wb-10008,nexus,01,22982,POINT (2035263.731 2454423.993)
4,nex-1001,wb-1001,nexus,01,None,POINT (2068060.445 2868197.372)
...,...,...,...,...,...,...
11251,cnx-5,wb-0,coastal,01,None,POINT (1946015.154 2272859.997)
11252,cnx-4,wb-0,coastal,01,None,POINT (1865508.752 2233140.002)
11253,cnx-3,wb-0,coastal,01,None,POINT (1874002.497 2238330.003)
11254,cnx-2,wb-0,coastal,01,None,POINT (1851141.345 2220479.997)


In [58]:
df_m_nex.loc[~df_m_nex['id'].isin(df_us_nex['id'])]

,id,toid,type,vpuid,poi_id,geometry
5196,nex-19437,wb-19437,nexus,01,NaN,POINT (1974761.175 2601253.199)


In [68]:
nexus = catalog.load_table(f"conus_hf.nexus").scan(row_filter=EqualTo("id","nex-1102857")).to_polars()
nexus

id,toid,type,vpuid,poi_id,geometry
str,str,str,str,f64,binary


In [60]:
df_us_da = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_us3/vpu02.gpkg", layer="divide-attributes")
df_m_da = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_r/vpus/02_nextgen.gpkg", layer="divide-attributes")

In [64]:
df_m_da.loc[~df_m_da['divide_id'].isin(df_us_da['divide_id'])]

,divide_id,mode.bexp_soil_layers_stag=1,mode.bexp_soil_layers_stag=2,mode.bexp_soil_layers_stag=3,mode.bexp_soil_layers_stag=4,mode.ISLTYP,mode.IVGTYP,geom_mean.dksat_soil_layers_stag=1,geom_mean.dksat_soil_layers_stag=2,geom_mean.dksat_soil_layers_stag=3,...,mean.Zmax,mode.Expon,centroid_x,centroid_y,mean.impervious,mean.elevation,mean.slope,circ_mean.aspect,dist_4.twi,vpuid
5294,cat-1e+05,7.323606,7.323606,7.323606,7.323606,6.0,3.0,0.000002,0.000002,0.000002,...,224.62338,4.0,1.555404e+06,2.014860e+06,0.578522,27090.643922,80.206922,203.840739,"[{""v"":4.009,""frequency"":0.25},{""v"":6.897,""freq...",02


In [66]:
df_us_n = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_us3/vpu07.gpkg", layer="nexus")
df_m_n = gpd.read_file("/home/quercus.hamlin/Documents/data/hydrofabric_data/vpus_r/vpus/07_nextgen.gpkg", layer="nexus")
df_m_n.loc[~df_m_n['id'].isin(df_us_n['id'])]

,id,toid,type,vpuid,poi_id,geometry
16078,nex-1102857,wb-1102857,nexus,07,NaN,POINT (514681.179 2424728.02)
